<a href="https://colab.research.google.com/github/mtryptnkr-study/Fake-News-Prediction/blob/main/P4_FakeNewsPredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fake News Predictor - Textual Data

- 1: Fake news
- 0: Real news

# Dataset Source
- https://www.kaggle.com/datasets/rahulogoel/isot-fake-news-dataset

# Methodology
1. Data Collection
2. Data pre-preocessing
3. Train-test split
4. Logistic Regression - Binary Classification
5. Evaluation of model on the basis of new data

# Importing Dependencies

In [76]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

- ```re``` - for searching text in document
- ```nltk.corpus``` - Natural Language Toolkit
- ```stopwords``` - words which do not value to the context of data (a, the, and)
- ```TfidfVectorizer``` - convert the text into feature vectors
- ```PorterStemmer``` - Stemming procedure

## Stopwords in English

In [77]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [78]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# Data Pre-processing

In [79]:
fake_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Machine Learning Projects/P4_Fake News Prediction/Dataset/Fake.csv')
true_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Machine Learning Projects/P4_Fake News Prediction/Dataset/True.csv')

In [80]:
fake_data.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


## Combining and shuffling the dataset

In [81]:
fake_data['label'] = 1
true_data['label'] = 0

In [82]:
fake_data.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [83]:
# Combine the datasets
combined_df = pd.concat([fake_data, true_data], ignore_index=True)

# Shuffle the records randomly
# frac=1 shuffles 100% of the rows
data = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [84]:
data.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0


In [85]:
data.shape

(44898, 5)

In [86]:
fake_data.shape

(23481, 5)

In [87]:
true_data.shape

(21417, 5)

## Checking missing values

In [88]:
print(data.isnull().sum())

title      0
text       0
subject    0
date       0
label      0
dtype: int64


## Using the ```title``` and ```text``` for prediction

In [89]:
# merging the title and text
data['content'] = data['title'] + ' ' + data['text']

In [90]:
print(data['content'])

0        Ben Stein Calls Out 9th Circuit Court: Committ...
1        Trump drops Steve Bannon from National Securit...
2        Puerto Rico expects U.S. to lift Jones Act shi...
3         OOPS: Trump Just Accidentally Confirmed He Le...
4        Donald Trump heads for Scotland to reopen a go...
                               ...                        
44893    UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...
44894    PM May seeks to ease Japan's Brexit fears duri...
44895    Merkel: Difficult German coalition talks can r...
44896     Trump Stole An Idea From North Korean Propaga...
44897    BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...
Name: content, Length: 44898, dtype: object


In [91]:
# separating the data & label
X = data.drop(columns='label', axis=1)
y = data['label']

In [92]:
X.head()

,title,text,subject,date,content
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",Ben Stein Calls Out 9th Circuit Court: Committ...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",Trump drops Steve Bannon from National Securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",Puerto Rico expects U.S. to lift Jones Act shi...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",OOPS: Trump Just Accidentally Confirmed He Le...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",Donald Trump heads for Scotland to reopen a go...


In [93]:
y.head()

,label
0,1
1,0
2,0
3,1
4,0


# Stemming Procedure
- Stemming is the process of reducing a word to its root word.
- Removes prefix and suffix.
  - Actor, Actress, Acting --> act is the root word

In [94]:
port_stem = PorterStemmer()

In [95]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

- We are creating a function called ```stemming```.
- We are calling regular expression library to search for text.
- ```sub``` substitutes certain values
- ```[^a-zA-Z]``` considers only the alphabets and removes numbers, punctuations etc. and replaces them by ' '
- We then convert all the alphabets to lower case
- Then we split it in list
- Then we take each word and reduce it to root word while removing the stopwords, so we use ```for``` loop to parse through
- After this we joined the words and return our stemmed content


In [96]:
data['content'] = data['content'].apply(stemming)

In [97]:
print(data['content'])

0        ben stein call th circuit court commit coup ta...
1        trump drop steve bannon nation secur council w...
2        puerto rico expect u lift jone act ship restri...
3        oop trump accident confirm leak isra intellig ...
4        donald trump head scotland reopen golf resort ...
                               ...                        
44893    unreal cb ted koppel tell sean hanniti bad ame...
44894    pm may seek eas japan brexit fear trade visit ...
44895    merkel difficult german coalit talk reach deal...
44896    trump stole idea north korean propaganda parod...
44897    break hillari clinton state depart gave russia...
Name: content, Length: 44898, dtype: object


# Splitting the data

In [98]:
# separating the data and label
X = data['content'].values
y = data['label'].values

In [99]:
print(X)

['ben stein call th circuit court commit coup tat constitut st centuri wire say ben stein reput professor pepperdin univers also hollywood fame appear tv show film ferri bueller day made provoc statement judg jeanin pirro show recent discuss halt impos presid trump execut order travel stein refer judgement th circuit court washington state coup tat execut branch constitut stein went call judg seattl polit puppet judiciari polit pawn watch interview complet statement note stark contrast rhetor leftist media pundit neglect note court ever block presidenti order immigr past discuss legal efficaci halt actual text execut order read trump news st centuri wire trump filessupport work subscrib becom member wire tv'
 'trump drop steve bannon nation secur council washington reuter u presid donald trump remov chief strategist steve bannon nation secur council wednesday revers controversi decis earli year give polit advis unpreced role secur discuss trump overhaul nsc confirm white hous offici al

In [100]:
print(y)

[1 0 0 ... 0 1 1]


# Converting the textual data to numerical data
- We need to convert all the text into computer understandable numbers
- ```TfidfVectorizer``` - Term Frequency Function
  - number of times the word is repeating and assigns a value to it

In [101]:
vetorizer = TfidfVectorizer()
vetorizer.fit(X)

X = vetorizer.transform(X)

In [102]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6904081 stored elements and shape (44898, 89868)>
  Coords	Values
  (0, 641)	0.05228664096628702
  (0, 2301)	0.031027142587997773
  (0, 3547)	0.0488071293292804
  (0, 6676)	0.04884347413276789
  (0, 6957)	0.16386262721973816
  (0, 8254)	0.06315631043962108
  (0, 9342)	0.08630781517552194
  (0, 10207)	0.17073096196903942
  (0, 11021)	0.0668114796652417
  (0, 12311)	0.13322555039706324
  (0, 13703)	0.17834547765750505
  (0, 14832)	0.058143818186916094
  (0, 14922)	0.05674091538137144
  (0, 15293)	0.12427662090796918
  (0, 15379)	0.08367337417927549
  (0, 15857)	0.1738925675982146
  (0, 15874)	0.1510751930350872
  (0, 17608)	0.03800104523939401
  (0, 19389)	0.10424208397556521
  (0, 21714)	0.14586212082964276
  (0, 23524)	0.05444062170305505
  (0, 23719)	0.16347523092935848
  (0, 24305)	0.10158376192651036
  (0, 24994)	0.12248771532966563
  (0, 25243)	0.08802206365616412
  :	:
  (44897, 66432)	0.11456982900715149
  (44897, 6701

# Splitting for training the model

In [103]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Predicting using Logistic Regression

In [104]:
model = LogisticRegression()

In [105]:
model.fit(X_train, y_train)

LogisticRegression()

## Evaluation

In [106]:
# accuracy on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, y_train)
print('Accuracy Score of training data: ', training_data_accuracy)

Accuracy Score of training data:  0.9911465003619355


In [107]:
# accuracy on test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, y_test)
print('Accuracy Score of test data: ', test_data_accuracy)

Accuracy Score of test data:  0.9857461024498887


# Predicting using RF Classifier

In [108]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [110]:
model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Evaluation

In [111]:
# accuracy on training data
X_train_prediction_RF = model2.predict(X_train)
training_data_accuracy_RF = accuracy_score(X_train_prediction_RF, y_train)
print('Accuracy Score of training data: ', training_data_accuracy_RF)

Accuracy Score of training data:  1.0


In [112]:
# accuracy on test data
X_test_prediction_RF = model2.predict(X_test)
test_data_accuracy_RF = accuracy_score(X_test_prediction_RF, y_test)
print('Accuracy Score of test data: ', test_data_accuracy_RF)

Accuracy Score of test data:  0.9934298440979955


In [113]:
print(classification_report(y_test, X_test_prediction_RF))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      4284
           1       1.00      0.99      0.99      4696

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [114]:
cm = confusion_matrix(y_test, X_test_prediction_RF)
print(cm)

[[4263   21]
 [  38 4658]]


# Making a Predictive System of the best model

In [115]:
X_new = X_test[0]

prediction = model2.predict(X_new)
print(prediction)

if prediction[0]==0:
  print('The news is Real')
else:
  print('The news is Fake')

[1]
The news is Fake


In [116]:
print(y_test[0])

1
